### Serve the PFD Obligations MCP server and chat app from a cluster

Runs `mcp_server/app.py` and `chat_app/app.py` (Streamlit) as background processes on this cluster's driver, reachable through the cluster's driver proxy. No Databricks Apps needed.

| Service | Port | Who uses it |
| --- | --- | --- |
| MCP server (`/mcp`) | 8000 | The chat app and external MCP clients, through the proxy with a token |
| Chat app (Streamlit) | 8501 | You, in the browser |

**Run order:** cells 2 to 8 (and 10 to smoke-test), then use the private runtime links described in cell 9. Use cell 11 for logs and cell 12 to stop.

**Requirements**
* A classic cluster with a driver proxy. Serverless is not supported.
* Set `DATABRICKS_WAREHOUSE_ID`, `DATABRICKS_SECRET_SCOPE`, and `DATABRICKS_TOKEN_SECRET_KEY` in the compute environment. The scope and key identify an existing secret containing the SQL warehouse PAT; do not put the token in source code. Optionally set `DATABRICKS_HOST` and `SERVING_ENDPOINT`.
* Workspace, organization, and cluster identifiers are discovered from runtime metadata without deployment-specific fallbacks.
* The chat app uses the notebook context token to reach the MCP server through the driver proxy. Everything runs as **you**, with your data, warehouse, and model endpoint permissions.
* Anyone opening the chat link needs attach permission on the cluster, and their questions run with **your** token.
* Do not bypass Databricks output redaction. Clear all outputs before saving or sharing this notebook, including service logs and smoke-test results.

The services stop when the cluster terminates, the notebook detaches, or Python restarts.

In [0]:
%pip install --quiet "mcp>=1.10,<2" "databricks-sql-connector>=4.0" "streamlit>=1.38" databricks-mcp openai "uvicorn>=0.52"

In [0]:
dbutils.library.restartPython()

In [0]:
import os, sys, tempfile

REPO_ROOT = os.getcwd()
MCP_DIR = os.path.join(REPO_ROOT, "mcp_server")
CHAT_DIR = os.path.join(REPO_ROOT, "chat_app")
assert os.path.isdir(MCP_DIR) and os.path.isdir(CHAT_DIR), "Expected mcp_server/ and chat_app/ under the current directory"

MCP_PORT = 8000
CHAT_PORT = 8501


def required_env(name):
    value = os.environ.get(name, "").strip()
    if not value:
        raise RuntimeError(f"Set {name} in the compute environment before running this notebook.")
    return value


WAREHOUSE_ID = required_env("DATABRICKS_WAREHOUSE_ID")
SECRET_SCOPE = required_env("DATABRICKS_SECRET_SCOPE")
TOKEN_SECRET_KEY = required_env("DATABRICKS_TOKEN_SECRET_KEY")
SERVING_ENDPOINT = os.environ.get("SERVING_ENDPOINT", "databricks-claude-sonnet-5")


def conf(key, default=None):
    try:
        return spark.conf.get(key)
    except Exception:
        return default


HOST = (os.environ.get("DATABRICKS_HOST") or conf("spark.databricks.workspaceUrl") or "").removeprefix("https://").rstrip("/")
ORG_ID = conf("spark.databricks.clusterUsageTags.clusterOwnerOrgId")
CLUSTER_ID = conf("spark.databricks.clusterUsageTags.clusterId")
if os.environ.get("IS_SERVERLESS", "").upper() == "TRUE":
    raise RuntimeError("Serverless has no driver proxy. Attach this notebook to a classic cluster.")
if not all((HOST, ORG_ID, CLUSTER_ID)):
    raise RuntimeError("Workspace or cluster metadata is unavailable. Attach this notebook to a classic cluster.")

TOKEN = dbutils.secrets.get(SECRET_SCOPE, TOKEN_SECRET_KEY)

try:
    CTX_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
except Exception as exc:
    raise RuntimeError("Cannot obtain a notebook API token for the driver proxy. Configure a proxy-authorized token before starting services; the scoped SQL PAT cannot be used here.") from exc

PROXY_API = f"https://{HOST}/driver-proxy-api/o/{ORG_ID}/{CLUSTER_ID}"
PROXY_WEB = f"https://{HOST}/driver-proxy/o/{ORG_ID}/{CLUSTER_ID}"

LOG_DIR = os.path.join(tempfile.gettempdir(), "mcp_rdg_logs")
os.makedirs(LOG_DIR, exist_ok=True)

BASE_ENV = {**os.environ, "DATABRICKS_HOST": f"https://{HOST}", "PYTHONUNBUFFERED": "1"}
for key in ("DATABRICKS_CLIENT_ID", "DATABRICKS_CLIENT_SECRET", "DATABRICKS_CONFIG_PROFILE", "DATABRICKS_AUTH_TYPE"):
    BASE_ENV.pop(key, None)

print("Cluster configuration loaded. Infrastructure identifiers are not printed.")

In [0]:
import os, signal, subprocess, time
import requests

SERVICES = globals().get("SERVICES", {})  # name -> {proc, log, port}


def port_open(port: int) -> bool:
    """True if something on the driver is listening on the port (checked with ss, not a loopback connect)."""
    out = subprocess.run(["ss", "-ltnH"], capture_output=True, text=True).stdout
    return any(line.split()[3].endswith(f":{port}") for line in out.splitlines() if len(line.split()) > 3)


def proxy_get(port: int, path: str = "/", timeout: int = 10) -> requests.Response:
    return requests.get(f"{PROXY_API}/{port}{path}", headers={"Authorization": f"Bearer {CTX_TOKEN}"}, timeout=timeout)


def tail(name: str, n: int = 40) -> None:
    path = SERVICES.get(name, {}).get("log")
    if not path or not os.path.exists(path):
        print(f"No log for {name}.")
        return
    lines = open(path, errors="replace").read().splitlines()
    print(f"--- last {min(n, len(lines))} lines of {name} log ({path}) ---")
    print("\n".join(lines[-n:]))


def start_service(name, cmd, cwd, env, port, health_path="/", timeout=90):
    svc = SERVICES.get(name)
    if svc and svc["proc"].poll() is None:
        print(f"{name} is already running (pid {svc['proc'].pid}).")
        return
    if port_open(port):
        print(f"Port {port} is already in use by another process (perhaps another notebook). Stop it or change the port.")
        return
    log_path = os.path.join(LOG_DIR, f"{name}.log")
    proc = subprocess.Popen(
        cmd, cwd=cwd, env=env, stdout=open(log_path, "w"), stderr=subprocess.STDOUT, start_new_session=True
    )
    SERVICES[name] = {"proc": proc, "log": log_path, "port": port}
    deadline = time.time() + timeout
    while time.time() < deadline:
        if proc.poll() is not None:
            print(f"{name} exited with code {proc.returncode}.")
            tail(name)
            return
        if port_open(port):
            try:
                r = proxy_get(port, health_path)
                if r.status_code == 200:
                    print(f"{name} is up on port {port} (pid {proc.pid}), health check through the proxy: HTTP 200.")
                    return
            except requests.RequestException:
                pass
        time.sleep(2)
    print(f"{name} did not answer within {timeout}s.")
    tail(name)


def stop_service(name: str) -> None:
    svc = SERVICES.get(name)
    if not svc or svc["proc"].poll() is not None:
        print(f"{name} is not running.")
        return
    os.killpg(svc["proc"].pid, signal.SIGTERM)
    try:
        svc["proc"].wait(timeout=10)
    except subprocess.TimeoutExpired:
        os.killpg(svc["proc"].pid, signal.SIGKILL)
    print(f"{name} stopped.")

In [0]:
start_service(
    "mcp_server",
    [sys.executable, "app.py"],
    cwd=MCP_DIR,
    env={**BASE_ENV, "DATABRICKS_TOKEN": TOKEN, "DATABRICKS_APP_PORT": str(MCP_PORT), "DATABRICKS_WAREHOUSE_ID": WAREHOUSE_ID},
    port=MCP_PORT,
    health_path="/",
)

In [0]:
# Hide the notebook runtime module from the child process. Otherwise WorkspaceClient() in app.py
# thinks it's inside a notebook, tries to build a SparkContext, and fails on every question.
STREAMLIT_LAUNCHER = (
    "import sys; sys.modules['dbruntime'] = None; "
    "from streamlit.web.cli import main; "
    "sys.argv = ['streamlit', 'run', 'app.py', *sys.argv[1:]]; sys.exit(main())"
)

# Streamlit checks files next to app.py (secrets.toml, pages/) on every request and session. From this
# subprocess, reads under /Workspace sometimes fail with PermissionError, which shows up as HTTP 500s.
# So run a local copy of chat_app/. Re-run this cell (after stopping the app) to pick up code changes.
import shutil
CHAT_RUN_DIR = os.path.join(tempfile.gettempdir(), "mcp_rdg_chat_app")
shutil.rmtree(CHAT_RUN_DIR, ignore_errors=True)
shutil.copytree(CHAT_DIR, CHAT_RUN_DIR, ignore=shutil.ignore_patterns(".venv", "__pycache__", ".git"))

start_service(
    "chat_app",
    [
        sys.executable, "-c", STREAMLIT_LAUNCHER,
        "--server.port", str(CHAT_PORT),
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--browser.gatherUsageStats", "false",
    ],
    cwd=CHAT_RUN_DIR,
    env={
        **BASE_ENV,
        "DATABRICKS_TOKEN": CTX_TOKEN,  # model endpoint + driver proxy
        "MCP_SERVER_URL": f"{PROXY_API}/{MCP_PORT}/mcp",  # reach the MCP server through the proxy
        "SERVING_ENDPOINT": SERVING_ENDPOINT,
    },
    port=CHAT_PORT,
    health_path="/_stcore/health",
)

In [0]:
CHAT_URL = f"{PROXY_WEB}/{CHAT_PORT}/"
MCP_URL = f"{PROXY_API}/{MCP_PORT}/mcp"

print("Chat and MCP URLs are available in CHAT_URL and MCP_URL for private use.")
print("The MCP endpoint requires a Bearer token authorized for the driver proxy.")
print("Do not save infrastructure URLs, tokens, or service logs in notebook outputs.")

**Private runtime links**

Use the `CHAT_URL` and `MCP_URL` variables computed above for the current cluster. No workspace or cluster URLs are stored in this notebook. View links only in a private session and clear all outputs before saving or sharing.

The MCP endpoint requires a Bearer token authorized for the driver proxy; never paste tokens into this notebook.

In [0]:
import json, requests

MCP_HEADERS = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream"}


def mcp_call(url, method, params=None, timeout=120):
    headers = {**MCP_HEADERS, "Authorization": f"Bearer {CTX_TOKEN}"}
    body = {"jsonrpc": "2.0", "id": 1, "method": method, "params": params or {}}
    r = requests.post(url, headers=headers, json=body, timeout=timeout)
    r.raise_for_status()
    return r.json()


tools = mcp_call(MCP_URL, "tools/list")["result"]["tools"]
print("MCP tools through the proxy:", [t["name"] for t in tools])

res = mcp_call(MCP_URL, "tools/call", {"name": "list_distinct_values", "arguments": {"column": "investment_type", "limit": 5}})
print("Tool call (queries the warehouse):", res["result"]["content"][0]["text"][:300])

r = proxy_get(CHAT_PORT, "/_stcore/health")
print("Chat app through the proxy:", r.status_code, r.text[:50])

In [0]:
for name, svc in SERVICES.items():
    state = "running" if svc["proc"].poll() is None else f"exited ({svc['proc'].returncode})"
    print(f"{name}: {state}, port {svc['port']}")
    tail(name, 30)
    print()

In [0]:
for name in list(SERVICES):
    stop_service(name)
print("Ports still open:", {p: port_open(p) for p in (MCP_PORT, CHAT_PORT)})